# Chapter 15: InSAR Processing

Complete InSAR chain from GRD amplitude proxy to deformation report.

In [ ]:
import sys; sys.path.insert(0,"../..")
from pygeovision.insar import InSARProcessor, InSARViz, InSARInterpreter
from pygeovision.insar.interferogram import generate_interferogram, amplitude_coherence
from pygeovision.insar.coherence import estimate_coherence, coherence_mask
from pygeovision.insar.displacement import phase_to_displacement, displacement_rate
print("InSAR modules loaded")

## InSARProcessor facade

In [ ]:
proc = InSARProcessor(
    output_dir          = "/tmp/pgv_insar_demo/",
    wavelength          = 0.05546576,   # Sentinel-1 C-band
    incidence_angle_deg = 39.0,
    coherence_threshold = 0.4,
)
print(proc)

## Synthetic demo

In [ ]:
import numpy as np, tempfile, rasterio
from rasterio.transform import from_bounds

def make_sar(path, H=64, W=64, val_range=(0.1,0.9)):
    data = np.random.uniform(*val_range, (H,W)).astype("float32")
    transform = from_bounds(-0.3, 5.5, -0.05, 5.7, W, H)
    profile = {"driver":"GTiff","dtype":"float32","count":1,"width":W,"height":H,
               "crs":rasterio.crs.CRS.from_epsg(4326),"transform":transform,"nodata":-9999.0}
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(data, 1)
    return path

import pathlib
tmp = pathlib.Path(tempfile.mkdtemp())
pre  = make_sar(str(tmp/"pre.tif"),  val_range=(0.3, 0.7))
post = make_sar(str(tmp/"post.tif"), val_range=(0.2, 0.8))  # more variation = change
print(f"Pre: {pre}\nPost: {post}")

## Generate interferogram

In [ ]:
ifg_result = proc.interferogram(pre, post, filter_type="none")
print(f"Interferogram: {ifg_result.interferogram_path}")
print(f"Coherence:     {ifg_result.coherence_path}")
print(f"Mean coherence:{ifg_result.stats.get('mean_coherence',0):.3f}")

## Displacement

In [ ]:
disp_result = proc.displacement(ifg_result, mode="amplitude_proxy")
print(f"Displacement: {disp_result.displacement_path}")
print(f"Max deformation: {disp_result.stats.get('max_displacement_m',0)*100:.2f} cm")

## Full pipeline

In [ ]:
full = proc.full_pipeline(pre, post, study_area="Demo Area")
print("\n" + full.report.summary())

## Visualization

In [ ]:
from pygeovision.insar.visualization import InSARViz
viz = InSARViz()
viz.displacement(full.displacement_path).show()
print("Dashboard available: viz.dashboard(...)")